# 00 Train Dog Pose Model on Colab

This notebook is for one job only: train a dog-pose checkpoint on Google Colab, then download `dog_pose_best.pt` back to your local machine.

Recommended flow:
1. Upload this notebook to Colab
2. Switch runtime to `GPU`
3. Run all cells
4. Download `dog_pose_best.pt`
5. Put it into your local repo at `models/dog_pose_best.pt`
6. Run the local pose-tagging notebook

## References

- Dog-Pose docs: https://docs.ultralytics.com/datasets/pose/dog-pose/
- Pose task docs: https://docs.ultralytics.com/tasks/pose/
- Dog-Pose dataset YAML: https://github.com/ultralytics/ultralytics/blob/main/ultralytics/cfg/datasets/dog-pose.yaml

Important: Ultralytics provides the dataset definition and the training route here. You are training your own dog-pose checkpoint from a general pose model initialization.

In [ ]:
!pip -q install -U ultralytics

In [ ]:
from pathlib import Path
import json
import shutil

import torch
from ultralytics import YOLO

if not torch.cuda.is_available():
    raise RuntimeError("Colab GPU is not enabled. Go to Runtime -> Change runtime type -> GPU.")

print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

## Training Config

The defaults below aim for a practical first model on Colab. If you want a quicker smoke test, reduce `EPOCHS` to `10` or `20`. If you want a stronger model, increase it later.

In [ ]:
BASE_MODEL_CANDIDATES = ["yolo11n-pose.pt", "yolov8n-pose.pt", "yolo26n-pose.pt"]
DATA_YAML = "dog-pose.yaml"
EPOCHS = 150
IMGSZ = 640
BATCH = 32
PROJECT_DIR = Path("/content/dog_pose_training")
RUN_NAME = "dog_pose_colab"
FINAL_MODEL_PATH = Path("/content/dog_pose_best.pt")
ZIP_OUTPUT_PATH = Path("/content/dog_pose_training_artifacts.zip")

PROJECT_DIR.mkdir(parents=True, exist_ok=True)

config_snapshot = {
    "BASE_MODEL_CANDIDATES": BASE_MODEL_CANDIDATES,
    "DATA_YAML": DATA_YAML,
    "EPOCHS": EPOCHS,
    "IMGSZ": IMGSZ,
    "BATCH": BATCH,
    "PROJECT_DIR": str(PROJECT_DIR),
    "RUN_NAME": RUN_NAME,
    "FINAL_MODEL_PATH": str(FINAL_MODEL_PATH),
}
print(json.dumps(config_snapshot, indent=2))

## Train Dog Pose

Ultralytics will download the official Dog-Pose dataset automatically from `dog-pose.yaml` if it is not already present in the Colab runtime.

In [ ]:
base_model_name = None
model = None
last_error = None

for candidate in BASE_MODEL_CANDIDATES:
    try:
        model = YOLO(candidate)
        base_model_name = candidate
        break
    except Exception as exc:
        last_error = exc

if model is None:
    raise RuntimeError(
        f"Unable to load any base pose model from {BASE_MODEL_CANDIDATES}. Last error: {last_error}"
    )

print("Training from base model:", base_model_name)

results = model.train(
    data=DATA_YAML,
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    device=0,
    project=str(PROJECT_DIR),
    name=RUN_NAME,
    exist_ok=True,
    cache=True,
    workers=8,
    pretrained=True,
    verbose=True,
)

## Collect the Best Checkpoint

This cell copies the training output into `/content/dog_pose_best.pt` and checks that the trained model exposes 24 dog keypoints.

In [ ]:
def find_best_checkpoint(model_obj):
    candidates = []
    trainer = getattr(model_obj, "trainer", None)
    if trainer is not None:
        best_path = getattr(trainer, "best", None)
        if best_path:
            candidates.append(Path(best_path))
        save_dir = getattr(trainer, "save_dir", None)
        if save_dir:
            candidates.append(Path(save_dir) / "weights" / "best.pt")
    candidates.append(PROJECT_DIR / RUN_NAME / "weights" / "best.pt")
    for candidate in candidates:
        if candidate.exists():
            return candidate
    return None


best_path = find_best_checkpoint(model)
if best_path is None:
    raise FileNotFoundError("Training finished, but best.pt was not found in the expected output paths.")

shutil.copy2(best_path, FINAL_MODEL_PATH)
print("Best checkpoint:", best_path)
print("Copied to:", FINAL_MODEL_PATH)

trained_model = YOLO(str(FINAL_MODEL_PATH))
kpt_shape = getattr(getattr(trained_model, "model", None), "kpt_shape", None)
print("Trained model kpt_shape:", kpt_shape)
if kpt_shape is not None and int(kpt_shape[0]) != 24:
    raise RuntimeError(f"Expected 24 dog keypoints, but got {kpt_shape}.")

## Download Outputs

The first cell downloads the trained checkpoint only. The second one optionally zips the whole training folder so you can keep metrics and plots too.

In [ ]:
from google.colab import files

files.download(str(FINAL_MODEL_PATH))

In [ ]:
archive_base = str(ZIP_OUTPUT_PATH).replace(".zip", "")
shutil.make_archive(archive_base, "zip", root_dir=PROJECT_DIR / RUN_NAME)
files.download(str(ZIP_OUTPUT_PATH))

## Next Step Back on Local

After download:
1. Create local folder `models/` if needed
2. Place the file at `models/dog_pose_best.pt`
3. Run the local pose-tagging notebook to build the pose index for `data/Images/`